# Lesson 96 · Whiteboard practice — close the laptop and derive the FFT, CTC and attention out loud

**Learning goals**
1. Derive the FFT butterfly formula on a whiteboard without looking at any code
2. Write out the CTC forward-probability recursion without looking at any code
3. Derive the matrix form of scaled dot-product attention without looking at any code
4. Find the blind spots in your derivations and write them down

> This is the only lesson in the whole course where you **may not run a code cell before you've finished the derivation**. The question types trace back to the whiteboard challenges in earlier lessons (FFT / CTC / attention).
> Work each question out on paper first; only once you're done does the "check your answers" cell unlock.

← **Previous lesson** · [L95 · Reading research papers](https://github.com/djzoom/AURORA/blob/main/notebooks/10_integration/L95_research_papers.ipynb) *(Chinese)*

> Last lesson covered **reading research papers**: the three-pass method, how papers are structured and written, the submission process, and academic collaboration.
> This lesson is **whiteboard practice**.

## Whiteboard rules

- Paper and pen; no notebook, no `src/`
- Time yourself: FFT ≤ 20 minutes, CTC ≤ 25 minutes, attention ≤ 15 minutes
- Only open the "check your answers" cell once you've finished the derivation — no peeking
- Keep a record: which step you got stuck on, and for how long

## 🎯 Don't panic: you've climbed these hills before

**None of today's three derivations is new material** — each one is a cliff you've already scaled yourself:

- **FFT butterfly** (L38-L39): divide and conquer folds O(N²) down to O(N log N)
- **CTC forward** (L68-L69): align sequences of different lengths with blanks, and sum path probabilities with dynamic programming
- **Attention** (L83): score with QKᵀ → softmax → weight V

Whiteboard practice doesn't test "have you learned this?" but "can you **close the laptop and reproduce it from memory**?" Getting stuck is normal — wherever you get stuck, go back to that lesson and derive it again.

## ✏️ Question 1: derive the FFT butterfly (target: under 20 minutes)

**The question**

Let the signal length be N = 8.

1. Write the definition of the DFT: X[k] = ?
2. Using the odd-even decomposition, write the Cooley-Tukey divide-and-conquer recursion
3. Draw the butterfly network for N=8 (8 inputs, 3 stages, 4 butterfly units per stage)
4. Write the two output formulas of one butterfly unit (in terms of the twiddle factor W_N^k)
5. Explain why the time complexity drops from O(N²) to O(N log N)

Finish it on paper first, then run the "check your answers" cell below.

In [ ]:
# Check-your-answers cell — run it only after you've finished the derivation
import numpy as np

# Definition of the DFT
def naive_dft(x):
    N = len(x)
    return np.array([sum(x[n] * np.exp(-2j*np.pi*k*n/N) for n in range(N))
                     for k in range(N)])

# Butterfly unit: takes E[k], O[k] and outputs two frequency components
def butterfly(E, O, k, N):
    twiddle = np.exp(-2j * np.pi * k / N)
    return E + twiddle * O, E - twiddle * O

# Verify: naive_dft vs np.fft.fft
x = np.random.randn(8)
assert np.allclose(naive_dft(x), np.fft.fft(x), atol=1e-9)
print('DFT matches the numpy reference ✅')
print()
print('Butterfly unit example (k=1, N=8):')
E, O = np.exp(1j*0.3), np.exp(1j*0.7)
a, b = butterfly(E, O, k=1, N=8)
print(f'  output +: {a:.4f}')
print(f'  output -: {b:.4f}')

In [ ]:
# ✏️ Self-review: record how your derivation went
fft_review = {
    "time_minutes":     None,   # how many minutes did it take you?
    "stuck_at":         None,   # which step did you get stuck on?
    "correct_parts":    None,   # which steps did you get right?
    "need_practice":    None,   # what needs more work next time?
}
print({k: v for k, v in fft_review.items()})

## ✏️ Question 2: the CTC forward algorithm (target: under 25 minutes)

**The question**

Let the target sequence be Y = [a, b]; after CTC expansion it becomes π = [-, a, -, b, -] (where - is the blank).

1. Write the definition of the CTC forward variable α_t(s)
2. Write the initial conditions α_1(1) and α_1(2)
3. Write the recursion (three cases: blank / repeated character / ordinary transition)
4. Explain why CTC doesn't need alignment labels

Finish it on paper first, then run the check-your-answers cell.

In [ ]:
# Check-your-answers cell — the core CTC forward recursion (simplified)
import numpy as np

def ctc_forward(log_probs, labels, blank=0):
    """
    log_probs: (T, V) log probability of each character at each frame
    labels:    target sequence (without blanks)
    returns:   log P(Y | X)
    """
    T, V = log_probs.shape
    # Expand the labels: insert a blank before and after every character
    l_prime = [blank]
    for c in labels:
        l_prime += [c, blank]
    S = len(l_prime)

    NEG_INF = float('-inf')
    alpha = np.full((T, S), NEG_INF)

    # Initialization
    alpha[0, 0] = log_probs[0, blank]
    if S > 1:
        alpha[0, 1] = log_probs[0, l_prime[1]]

    # Recursion
    for t in range(1, T):
        for s in range(S):
            c = l_prime[s]
            # from the same position
            a = alpha[t-1, s]
            # from the previous position
            if s > 0:
                a = np.logaddexp(a, alpha[t-1, s-1])
            # skip over a blank (current is not blank and differs from the one two back)
            if s > 1 and c != blank and c != l_prime[s-2]:
                a = np.logaddexp(a, alpha[t-1, s-2])
            alpha[t, s] = a + log_probs[t, c]

    return np.logaddexp(alpha[T-1, S-1], alpha[T-1, S-2])

# A simple test
T, V = 5, 3
np.random.seed(42)
raw = np.random.randn(T, V)
log_probs = raw - np.log(np.sum(np.exp(raw), axis=1, keepdims=True))
labels = [1, 2]  # target: character 1 then character 2
try:
    log_p = ctc_forward(log_probs, labels, blank=0)
    assert log_p is not None, "⬜ ctc_forward is not implemented"
    assert np.isfinite(log_p), f"the log probability should be finite, got {log_p}"
    assert log_p < 0, f"the log probability should be negative, got {log_p:.4f}"
    print(f'✅ CTC forward check passed, log P(Y|X) = {log_p:.4f}')
except (NotImplementedError, TypeError):
    print('⬜ ctc_forward is not implemented')

In [ ]:
ctc_review = {
    "time_minutes":  None,
    "stuck_at":      None,
    "correct_parts": None,
    "need_practice": None,
}
print(ctc_review)

## ✏️ Question 3: scaled dot-product attention (target: under 15 minutes)

**The question**

1. Write the full formula for Attention(Q, K, V)
2. Explain why we divide by √d_k (and not by d_k, or by 1)
3. What shape is the matrix you get after the softmax?
4. How does multi-head attention differ from single-head attention (one sentence)?

Finish it on paper first, then run the check-your-answers cell.

In [ ]:
import numpy as np

def scaled_dot_product_attention(Q, K, V):
    """
    Q: (seq_q, d_k)
    K: (seq_k, d_k)
    V: (seq_k, d_v)
    returns: (seq_q, d_v)
    """
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)          # (seq_q, seq_k)
    # softmax (numerically stable version)
    scores -= scores.max(axis=-1, keepdims=True)
    weights = np.exp(scores)
    weights /= weights.sum(axis=-1, keepdims=True)  # (seq_q, seq_k)
    return weights @ V                         # (seq_q, d_v)

# Check the shapes
seq_q, seq_k, d_k, d_v = 4, 6, 8, 16
np.random.seed(0)
Q = np.random.randn(seq_q, d_k)
K = np.random.randn(seq_k, d_k)
V = np.random.randn(seq_k, d_v)
out = scaled_dot_product_attention(Q, K, V)
assert out.shape == (seq_q, d_v)
print(f'output shape: {out.shape}  ✅')
print(f'why divide by √d_k: d_k={d_k} → divide by {d_k**0.5:.2f}, so large dot products don\'t saturate the softmax')

In [ ]:
attention_review = {
    "time_minutes":  None,
    "stuck_at":      None,
    "correct_parts": None,
    "need_practice": None,
}
print(attention_review)

## ✏️ Question 4: the capstone — the MFCC pipeline (target: under 30 minutes)

This question tests whether you can chain several modules together, not just derive one piece in isolation.

**The question**

Starting from a 16 kHz PCM signal x captured from a microphone and ending with 13-dimensional MFCC coefficients,
write the complete pipeline on the whiteboard (no code needed — just formulas and shapes):

1. Framing: write the formulas for the number of frames T and the frame length L (take window=25ms, hop=10ms)
2. Windowing: which window function? Write the expression for the Hann window
3. FFT: what shape is the output? Complex or real?
4. Power spectrum: write the formula that takes you from the FFT output to the power spectrum
5. Mel filtering: write the Hz→Mel mapping; what shape is the filterbank matrix?
6. Taking the log: why take the log?
7. DCT-II: write the DCT-II formula; why a DCT rather than another FFT?
8. Slicing: which coefficients come out at the end? Why drop coefficient 0?

Finish it on paper first, then run the check-your-answers cell.

In [ ]:
# Check-your-answers cell: shapes and formulas at each step of the MFCC pipeline
import numpy as np

sr = 16000
duration = 1.0
x = np.random.randn(int(sr * duration))   # simulate 1 second of PCM

# 1. Framing parameters
win_len = int(0.025 * sr)   # 25ms → 400 samples
hop     = int(0.010 * sr)   # 10ms → 160 samples
n_frames = 1 + (len(x) - win_len) // hop
print(f"1. Framing: {n_frames} frames × {win_len} samples/frame")

# 2. Hann window
hann = 0.5 * (1 - np.cos(2 * np.pi * np.arange(win_len) / (win_len - 1)))
print(f"2. Hann window shape: {hann.shape}")

# 3. FFT (keep the first half)
n_fft = win_len
n_freq = n_fft // 2 + 1
print(f"3. FFT output: {n_freq} complex frequency bins (keep the first {n_freq})")

# 4. Power spectrum
frame0 = x[:win_len] * hann
# Note: aurora.audio.fft implements an equivalent FFT; np.fft.rfft is borrowed here only to show the shape.
# In production code use Aurora's own implementation, in keeping with the "No black boxes" rule.
power = np.abs(np.fft.rfft(frame0, n=n_fft)) ** 2
print(f"4. Power spectrum shape: {power.shape}")

# 5. Mel filtering
n_mels = 40
def hz_to_mel(f): return 2595 * np.log10(1 + f / 700)
def mel_to_hz(m): return 700 * (10 ** (m / 2595) - 1)
print(f"5. Mel filterbank shape: ({n_mels}, {n_freq})")
print(f"   Hz→Mel: 2595 × log10(1 + f/700)")

# 6. Log
print("6. log(energy): compresses the dynamic range so low-energy regions stay distinguishable (human loudness perception is roughly logarithmic)")

# 7. DCT-II implementation (by hand, orthonormal norm='ortho', same convention as L49 / aurora.audio.mfcc.dct_ii)
def dct2(x):
    """DCT-II (orthonormal): X[k] = s_k · Σ_{n=0}^{M-1} x[n] · cos(π·k·(2n+1) / (2M))
    s_0 = √(1/M), s_k = √(2/M) (k>0) — an orthogonal matrix: the inverse is the transpose, and energy is preserved.
    """
    M = len(x)
    ns = np.arange(M)
    scale = np.full(M, np.sqrt(2.0 / M))
    scale[0] = np.sqrt(1.0 / M)
    return np.array([
        scale[k] * np.sum(x * np.cos(np.pi * k * (2 * ns + 1) / (2 * M)))
        for k in range(M)
    ])

# Check DCT-II on a fake Mel energy vector (matching scipy's norm='ortho')
fake_mel_log = np.log(np.abs(np.random.randn(n_mels)) + 1e-8)
cepstrum = dct2(fake_mel_log)
try:
    from scipy.fft import dct as scipy_dct
    ref = scipy_dct(fake_mel_log, type=2, norm='ortho')
    assert np.allclose(cepstrum, ref, atol=1e-9), "DCT-II does not match the scipy reference"
    print(f"7. DCT-II check passed (vs scipy.fft.dct, norm='ortho') ✅  output shape: {cepstrum.shape}")
except ImportError:
    print(f"7. DCT-II implemented (scipy not installed, cross-check skipped) shape: {cepstrum.shape}")

# 8. Slicing: keep coefficients 1-13 and drop coefficient 0 (the mean energy — tied to loudness, carries no timbre information)
n_mfcc = 13
mfcc_frame = cepstrum[1:n_mfcc + 1]
print(f"8. MFCC shape per frame after slicing: {mfcc_frame.shape}  (coefficients 1-{n_mfcc})")

## ✏️ Overall self-review: once all four questions are done

When you've finished all the derivations (the three core questions plus the capstone), do one complete self-review:
find the modules that still need work and review them in a targeted way, so your foundations become truly solid and hold up under follow-up questions.

In [ ]:
# Overall self-review (based on how the first three questions and the capstone went)
overall_review = {
    # Rate each module: 'proficient' / 'basic grasp' / 'needs work'
    "FFT butterfly recursion":           None,
    "CTC forward recursion":           None,
    "attention matrix derivation":     None,
    "full MFCC pipeline":          None,

    # The question that took you longest
    "took_longest":               None,   # e.g. "CTC"

    # The module you most need to derive again before an interview
    "review_first":               None,

    # Target for next time (e.g. "get the FFT from 25 minutes down to 15")
    "next_target":               None,
}

filled = sum(1 for v in overall_review.values() if v is not None)
n_proficient = sum(1 for k in ["FFT butterfly recursion","CTC forward recursion","attention matrix derivation","full MFCC pipeline"]
                   if overall_review.get(k) == "proficient")

print(f"filled in {filled}/{len(overall_review)} items")
print(f"proficient modules: {n_proficient}/4")
if overall_review["review_first"]:
    print(f"review first: {overall_review['review_first']}")
if overall_review["next_target"]:
    print(f"target for next time: {overall_review['next_target']}")

## Wrap-up

A whiteboard derivation is the most direct way to tell whether you **really understand** something or have only **memorized** it. If you ran over time or got stuck on a question:

1. Go back to the matching lessons (FFT → L38-L39, CTC → L68-L69, attention → L83)
2. Close the notebook and derive it again
3. Repeat until you can write it out from scratch in under 20 minutes

**Next lesson L97**: presenting your work — how to explain any Aurora module clearly to someone else, in a 30-second or a 3-minute version (an interview is just one of the occasions).

---

→ **Next lesson** · [L97 · Presenting your work](L97_interview.ipynb)

> Next lesson covers **presenting your work**: the "golden structure" of a technical interview (problem → method → implementation → result → limitations), a 30-second and a 3-minute version for every Aurora core module, and how to answer "why build it from scratch instead of using librosa?".